# Collect Data

In [75]:
%pip install pandas numpy seaborn matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [76]:
import pandas as pd
import numpy as np


In [77]:
import time
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [78]:
df = pd.read_csv('final_data_test.csv')
df['Periodo'] = pd.to_datetime(df['Periodo'], format='%Y-%m-%d')
df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,Densidade_GLP_Liquido,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29882,7-XA-4D-CES,2019-06-30,Ceará,Ceará,94.1782,251.5236,0.2339,717.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14097,1424.4825
29883,7-XA-4D-CES,2019-07-31,Ceará,Ceará,93.5297,167.4319,0.3863,742.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14128,1388.7033
29884,7-XA-4D-CES,2019-08-31,Ceará,Ceará,88.4686,141.8379,0.2658,737.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14159,1287.9177
29885,7-XA-4D-CES,2019-09-30,Ceará,Ceará,103.8475,95.3443,0.2822,709.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14189,1455.1776


In [79]:
df = df[df['Petroleo'] > 0]
df = df[df['Agua'] > 0]
df = df[df['Gas'] > 0]
df = df[df['Lamina_DAgua'] > 0]
df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,Densidade_GLP_Liquido,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29882,7-XA-4D-CES,2019-06-30,Ceará,Ceará,94.1782,251.5236,0.2339,717.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14097,1424.4825
29883,7-XA-4D-CES,2019-07-31,Ceará,Ceará,93.5297,167.4319,0.3863,742.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14128,1388.7033
29884,7-XA-4D-CES,2019-08-31,Ceará,Ceará,88.4686,141.8379,0.2658,737.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14159,1287.9177
29885,7-XA-4D-CES,2019-09-30,Ceará,Ceará,103.8475,95.3443,0.2822,709.0,28.1,2.039707,532.790002,38864.6256,-3.030599,-39.042199,0.0,31.5,XARÉU,14189,1455.1776


In [80]:
df = df[df['Estado'] == 'Rio de Janeiro']
df = df[df['Bacia'] == 'Campos']
df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,Densidade_GLP_Liquido,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.20,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.20,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.20,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.20,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.20,2.007566,528.948721,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29853,7-VM-14D-RJS,2025-08-31,Rio de Janeiro,Campos,123.0542,2475.7504,0.4162,432.0,22.86,2.044230,533.350020,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14540,2048.8904
29854,7-VM-14D-RJS,2025-09-30,Rio de Janeiro,Campos,99.5146,2192.6278,0.4992,408.0,22.86,2.044230,533.350020,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14570,2005.4093
29855,7-VM-14D-RJS,2025-10-31,Rio de Janeiro,Campos,67.0630,1098.4849,0.7693,230.0,22.86,2.044230,533.350020,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14601,1926.1265
29856,7-VM-14D-RJS,2025-11-30,Rio de Janeiro,Campos,136.2777,2616.5610,1.2561,481.0,22.86,2.044230,533.350020,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14631,1820.0921


In [81]:
from sklearn.ensemble import IsolationForest

features = df.drop(columns=['Nome_Poco_ANP', 'Periodo', 'Estado', 'Bacia', 'Latitude', 'Longitude', 'Campo', 'Profundidade'])
features = features.dropna()

iso_forest = IsolationForest(random_state=42)
iso_forest.fit(features)

df = df.loc[features.index].copy()
df['anomaly_score'] = iso_forest.decision_function(features)
df['anomaly'] = iso_forest.predict(features)

df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.20,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772,0.037952,1
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.20,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618,0.043747,1
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.20,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000,0.043747,1
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.20,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596,0.037014,1
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.20,2.007566,...,41614.2601,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609,0.037483,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29853,7-VM-14D-RJS,2025-08-31,Rio de Janeiro,Campos,123.0542,2475.7504,0.4162,432.0,22.86,2.044230,...,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14540,2048.8904,-0.022612,-1
29854,7-VM-14D-RJS,2025-09-30,Rio de Janeiro,Campos,99.5146,2192.6278,0.4992,408.0,22.86,2.044230,...,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14570,2005.4093,-0.025901,-1
29855,7-VM-14D-RJS,2025-10-31,Rio de Janeiro,Campos,67.0630,1098.4849,0.7693,230.0,22.86,2.044230,...,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14601,1926.1265,-0.044298,-1
29856,7-VM-14D-RJS,2025-11-30,Rio de Janeiro,Campos,136.2777,2616.5610,1.2561,481.0,22.86,2.044230,...,38122.0594,-22.189655,-40.303273,0.0,81.0,VERMELHO,14631,1820.0921,-0.021413,-1


In [82]:
sample_df = df[df['anomaly'] == 1]
sample_df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,PCS,Latitude,Longitude,Profundidade,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,...,41614.26010,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2774,1210.1772,0.037952,1
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,...,41614.26010,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2894,1538.6618,0.043747,1
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,...,41614.26010,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2924,1562.2000,0.043747,1
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,...,41614.26010,-22.031677,-39.943771,2623.5,626.0,ALBACORA,2955,1600.4596,0.037014,1
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,...,41614.26010,-22.031677,-39.943771,2623.5,626.0,ALBACORA,3016,1807.4609,0.037483,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29483,7-TVD-8D-RJS,2025-08-31,Rio de Janeiro,Campos,3072.1276,1629.6683,23.6395,598.0,26.8,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.6,910.0,TARTARUGA VERDE,792,2148.5860,0.005286,1
29484,7-TVD-8D-RJS,2025-09-30,Rio de Janeiro,Campos,2824.0924,1555.7035,21.2706,587.0,26.8,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.6,910.0,TARTARUGA VERDE,822,2106.3811,0.009034,1
29485,7-TVD-8D-RJS,2025-10-31,Rio de Janeiro,Campos,2915.1964,1605.8897,20.1178,611.0,26.8,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.6,910.0,TARTARUGA VERDE,853,2016.3880,0.006435,1
29486,7-TVD-8D-RJS,2025-11-30,Rio de Janeiro,Campos,3009.8116,1658.0103,19.2372,584.0,26.8,1.987930,...,42385.04465,-22.983486,-40.735555,-2982.6,910.0,TARTARUGA VERDE,883,1930.0290,0.003925,1


In [83]:
df_pocos = df[['Nome_Poco_ANP', 'Latitude', 'Longitude']]
df_pocos = df_pocos.drop_duplicates().dropna()
df_pocos

,Nome_Poco_ANP,Latitude,Longitude
0,7-AB-103HP-RJS,-22.031677,-39.943771
45,7-AB-108HP-RJS,-22.047854,-39.950322
74,7-AB-110HP-RJS,-22.062486,-39.948515
113,7-AB-114H-RJS,-22.063648,-39.951252
139,7-AB-117H-RJS,-22.048518,-39.938543
...,...,...,...
29246,7-TVD-15D-RJS,-22.944455,-40.693041
29326,7-TVD-16DP-RJS,-22.959105,-40.698762
29400,7-TVD-5D-RJS,-22.962363,-40.709910
29461,7-TVD-8D-RJS,-22.983486,-40.735555


In [84]:
from sklearn.metrics.pairwise import haversine_distances

coords_rad = np.radians(df_pocos[['Latitude', 'Longitude']].values)
dist_matrix_km = haversine_distances(coords_rad) * 6371
dist_matrix_km

array([[  0.        ,   1.92141625,   3.460532  , ..., 130.01675723,
        133.47902298,  40.98982176],
       [  1.92141625,   0.        ,   1.63759942, ..., 128.17596887,
        131.64096276,  39.62973951],
       [  3.460532  ,   1.63759942,   0.        , ..., 127.00083285,
        130.47130883,  39.18283035],
       ...,
       [130.01675723, 128.17596887, 127.00083285, ...,   0.        ,
          3.52275274,  95.52778853],
       [133.47902298, 131.64096276, 130.47130883, ...,   3.52275274,
          0.        ,  98.79875445],
       [ 40.98982176,  39.62973951,  39.18283035, ...,  95.52778853,
         98.79875445,   0.        ]], shape=(326, 326))

In [85]:
df_dist_c = pd.DataFrame(dist_matrix_km, index=df_pocos['Nome_Poco_ANP'], columns=df_pocos['Nome_Poco_ANP'])
df_dist_c

Nome_Poco_ANP,7-AB-103HP-RJS,7-AB-108HP-RJS,7-AB-110HP-RJS,7-AB-114H-RJS,7-AB-117H-RJS,7-AB-130HP-RJS,7-AB-132HP-RJS,7-AB-134HPA-RJS,7-AB-59H-RJS,7-AB-63HA-RJS,...,7-TBMT-8H-RJS,7-TMT-1D-RJS,7-TVD-10D-RJS,7-TVD-11D-RJS,7-TVD-12D-RJS,7-TVD-15D-RJS,7-TVD-16DP-RJS,7-TVD-5D-RJS,7-TVD-8D-RJS,7-VM-14D-RJS
Nome_Poco_ANP,,,,,,,,,,,,,,,,,,,,,
7-AB-103HP-RJS,0.000000,1.921416,3.460532,3.637731,1.948653,8.533558,7.447022,8.308231,12.302177,12.312367,...,170.087299,132.486606,134.525703,125.718488,126.399182,127.385860,129.037535,130.016757,133.479023,40.989822
7-AB-108HP-RJS,1.921416,0.000000,1.637599,1.758841,1.216240,8.178672,7.500271,8.299616,10.380980,10.391202,...,168.320420,130.645651,132.687308,123.871081,124.551054,125.544389,127.193829,128.175969,131.640963,39.629740
7-AB-110HP-RJS,3.460532,1.637599,0.000000,0.310285,1.862333,7.463154,7.161243,7.870706,8.968880,8.980102,...,167.280735,129.470023,131.516902,122.682132,123.360503,124.367977,126.012601,127.000833,130.471309,39.182830
7-AB-114H-RJS,3.637731,1.758841,0.310285,0.000000,2.132102,7.708527,7.446429,8.146984,8.729701,8.740671,...,166.991306,129.194981,131.241212,122.408777,123.087362,124.093013,125.738259,126.725718,130.195521,38.873096
7-AB-117H-RJS,1.948653,1.216240,1.862333,2.132102,0.000000,7.058277,6.298627,7.108137,10.819554,10.831007,...,169.110562,131.327078,133.373308,124.540677,125.219218,126.225106,127.870308,128.857817,132.327613,40.717221
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7-TVD-15D-RJS,127.385860,125.544389,124.367977,124.093013,126.225106,127.581865,128.970472,129.026516,115.588443,115.575117,...,45.890635,5.102457,7.177833,2.169816,1.848677,0.000000,1.731141,2.636033,6.146772,92.983451
7-TVD-16DP-RJS,129.037535,127.193829,126.012601,125.738259,127.870308,129.195426,130.590264,130.642165,117.224764,117.211479,...,44.630452,3.511980,5.634345,3.434346,2.842207,1.731141,0.000000,1.197528,4.640965,94.705920
7-TVD-5D-RJS,130.016757,128.175969,127.000833,126.725718,128.857817,130.217879,131.606454,131.662549,118.222957,118.209624,...,43.446654,2.470086,4.545597,4.574715,4.015690,2.636033,1.197528,0.000000,3.522753,95.527789


In [86]:
lista_distancias_finais = []

for poco in df_dist_c.index:
    vizinhos_ordenados = df_dist_c.loc[poco].sort_values()
    
    dist_v1 = vizinhos_ordenados.iloc[1]
    dist_v2 = vizinhos_ordenados.iloc[2]
    dist_v3 = vizinhos_ordenados.iloc[3]
    
    raio_limite = 2.0 
    pocos_no_raio = np.sum(vizinhos_ordenados.iloc[1:] <= raio_limite)
    
    lista_distancias_finais.append({
        'Nome_Poco_ANP': poco,
        'DIST_VIZINHO_1': dist_v1,
        'DIST_VIZINHO_2': dist_v2,
        'DIST_VIZINHO_3': dist_v3,
        'POCOS_NO_RAIO': pocos_no_raio
    })

df_features_espaciais = pd.DataFrame(lista_distancias_finais)
df_features_espaciais

,Nome_Poco_ANP,DIST_VIZINHO_1,DIST_VIZINHO_2,DIST_VIZINHO_3,POCOS_NO_RAIO
0,7-AB-103HP-RJS,1.921416,1.948653,3.460532,2
1,7-AB-108HP-RJS,1.216240,1.637599,1.758841,4
2,7-AB-110HP-RJS,0.310285,1.637599,1.862333,3
3,7-AB-114H-RJS,0.310285,1.758841,2.132102,2
4,7-AB-117H-RJS,1.216240,1.862333,1.948653,3
...,...,...,...,...,...
321,7-TVD-15D-RJS,1.731141,1.848677,2.169816,2
322,7-TVD-16DP-RJS,1.197528,1.731141,2.842207,2
323,7-TVD-5D-RJS,1.197528,2.470086,2.636033,1
324,7-TVD-8D-RJS,1.049545,1.209194,3.522753,2


In [87]:
sample_df = pd.merge(
    sample_df,
    df_features_espaciais,
    on='Nome_Poco_ANP',
    how='left'
    )
sample_df

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly,DIST_VIZINHO_1,DIST_VIZINHO_2,DIST_VIZINHO_3,POCOS_NO_RAIO
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,...,626.0,ALBACORA,2774,1210.1772,0.037952,1,1.921416,1.948653,3.460532,2
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,...,626.0,ALBACORA,2894,1538.6618,0.043747,1,1.921416,1.948653,3.460532,2
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,...,626.0,ALBACORA,2924,1562.2000,0.043747,1,1.921416,1.948653,3.460532,2
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,...,626.0,ALBACORA,2955,1600.4596,0.037014,1,1.921416,1.948653,3.460532,2
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,...,626.0,ALBACORA,3016,1807.4609,0.037483,1,1.921416,1.948653,3.460532,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12910,7-TVD-8D-RJS,2025-08-31,Rio de Janeiro,Campos,3072.1276,1629.6683,23.6395,598.0,26.8,1.987930,...,910.0,TARTARUGA VERDE,792,2148.5860,0.005286,1,1.049545,1.209194,3.522753,2
12911,7-TVD-8D-RJS,2025-09-30,Rio de Janeiro,Campos,2824.0924,1555.7035,21.2706,587.0,26.8,1.987930,...,910.0,TARTARUGA VERDE,822,2106.3811,0.009034,1,1.049545,1.209194,3.522753,2
12912,7-TVD-8D-RJS,2025-10-31,Rio de Janeiro,Campos,2915.1964,1605.8897,20.1178,611.0,26.8,1.987930,...,910.0,TARTARUGA VERDE,853,2016.3880,0.006435,1,1.049545,1.209194,3.522753,2
12913,7-TVD-8D-RJS,2025-11-30,Rio de Janeiro,Campos,3009.8116,1658.0103,19.2372,584.0,26.8,1.987930,...,910.0,TARTARUGA VERDE,883,1930.0290,0.003925,1,1.049545,1.209194,3.522753,2


In [88]:
sample_df['Petroleo'].describe()

count    12915.000000
mean      2481.806636
std       2025.733086
min          0.010900
25%       1095.432150
50%       1971.685600
75%       3209.500050
max      14500.866400
Name: Petroleo, dtype: float64

In [89]:
target_column = 'Petroleo'
ignore_columns = [
    'Nome_Poco_ANP', 'Latitude', 'Longitude', 'Campo', 'Estado', 'Bacia', 'Periodo', 'Profundidade', 'anomaly_score', 'anomaly'
]

In [90]:
# sample_df = df.sample(frac=1, random_state=42)
# sample_df.drop_duplicates(inplace=True)
# columns = [col for col in sample_df.columns if col not in ignore_columns]
# for col in columns:
#     Q1 = sample_df[col].quantile(0.25)
#     Q3 = sample_df[col].quantile(0.75)
#     IQR = Q3 - Q1

#     lower_bound = Q1 - 1.5 * IQR
#     upper_bound = Q3 + 1.5 * IQR

#     sample_df = sample_df[(sample_df[col] >= lower_bound) & (sample_df[col] <= upper_bound)]

# sample_df

In [91]:
train = sample_df[sample_df['Periodo'] < pd.to_datetime('2024-01-01')]
test = sample_df[sample_df['Periodo'] >= pd.to_datetime('2024-01-01')]

In [92]:
train

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly,DIST_VIZINHO_1,DIST_VIZINHO_2,DIST_VIZINHO_3,POCOS_NO_RAIO
0,7-AB-103HP-RJS,2018-01-31,Rio de Janeiro,Campos,1369.6102,15.4729,8.8017,677.0,26.2,2.007566,...,626.0,ALBACORA,2774,1210.1772,0.037952,1,1.921416,1.948653,3.460532,2
1,7-AB-103HP-RJS,2018-05-31,Rio de Janeiro,Campos,1387.1269,14.8405,12.3111,725.0,26.2,2.007566,...,626.0,ALBACORA,2894,1538.6618,0.043747,1,1.921416,1.948653,3.460532,2
2,7-AB-103HP-RJS,2018-06-30,Rio de Janeiro,Campos,1293.5024,30.2712,12.9338,720.0,26.2,2.007566,...,626.0,ALBACORA,2924,1562.2000,0.043747,1,1.921416,1.948653,3.460532,2
3,7-AB-103HP-RJS,2018-07-31,Rio de Janeiro,Campos,740.0016,11.5824,8.2655,484.0,26.2,2.007566,...,626.0,ALBACORA,2955,1600.4596,0.037014,1,1.921416,1.948653,3.460532,2
4,7-AB-103HP-RJS,2018-09-30,Rio de Janeiro,Campos,611.2813,2.4729,7.1527,332.0,26.2,2.007566,...,626.0,ALBACORA,3016,1807.4609,0.037483,1,1.921416,1.948653,3.460532,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12888,7-TVD-5D-RJS,2023-08-31,Rio de Janeiro,Campos,7455.4895,1092.3278,76.1519,744.0,26.8,1.997850,...,941.0,TARTARUGA VERDE,1231,2419.4681,0.016739,1,1.197528,2.470086,2.636033,1
12889,7-TVD-5D-RJS,2023-09-30,Rio de Janeiro,Campos,7327.0160,1084.6882,76.5751,718.0,26.8,1.997850,...,941.0,TARTARUGA VERDE,1261,2700.6249,0.011067,1,1.197528,2.470086,2.636033,1
12890,7-TVD-5D-RJS,2023-10-31,Rio de Janeiro,Campos,5952.7208,1687.2829,65.3762,684.0,26.8,1.997850,...,941.0,TARTARUGA VERDE,1292,2677.4448,0.017562,1,1.197528,2.470086,2.636033,1
12891,7-TVD-5D-RJS,2023-11-30,Rio de Janeiro,Campos,5987.0127,1826.6325,73.8396,720.0,26.8,2.007620,...,941.0,TARTARUGA VERDE,1322,2349.3362,0.016787,1,1.197528,2.470086,2.636033,1


In [93]:
test

,Nome_Poco_ANP,Periodo,Estado,Bacia,Petroleo,Agua,Gas,Tempo_Producao,Grau_API,Densidade_GLP_Gas,...,Lamina_DAgua,Campo,Idade,Preco_referencia,anomaly_score,anomaly,DIST_VIZINHO_1,DIST_VIZINHO_2,DIST_VIZINHO_3,POCOS_NO_RAIO
202,7-AB-134HPA-RJS,2024-01-31,Rio de Janeiro,Campos,10981.5097,766.5725,136.0015,723.0,22.85,2.066945,...,1015.0,ALBACORA,1938,2091.9442,0.014238,1,0.874132,1.508838,2.029951,2
203,7-AB-134HPA-RJS,2024-02-29,Rio de Janeiro,Campos,11732.4494,1108.8803,149.3063,687.0,22.85,2.066945,...,1015.0,ALBACORA,1967,2184.2019,0.004795,1,0.874132,1.508838,2.029951,2
204,7-AB-134HPA-RJS,2024-03-31,Rio de Janeiro,Campos,11073.3941,1046.5905,156.7299,742.0,22.85,2.066945,...,1015.0,ALBACORA,1998,2219.8154,0.009697,1,0.874132,1.508838,2.029951,2
205,7-AB-134HPA-RJS,2024-04-30,Rio de Janeiro,Campos,10836.5495,1707.7264,151.6220,720.0,22.85,2.066945,...,1015.0,ALBACORA,2028,2428.1163,0.005670,1,0.874132,1.508838,2.029951,2
206,7-AB-134HPA-RJS,2024-05-31,Rio de Janeiro,Campos,10376.2290,1649.5664,143.3321,744.0,22.85,2.066945,...,1015.0,ALBACORA,2059,2246.6465,0.014869,1,0.874132,1.508838,2.029951,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12910,7-TVD-8D-RJS,2025-08-31,Rio de Janeiro,Campos,3072.1276,1629.6683,23.6395,598.0,26.80,1.987930,...,910.0,TARTARUGA VERDE,792,2148.5860,0.005286,1,1.049545,1.209194,3.522753,2
12911,7-TVD-8D-RJS,2025-09-30,Rio de Janeiro,Campos,2824.0924,1555.7035,21.2706,587.0,26.80,1.987930,...,910.0,TARTARUGA VERDE,822,2106.3811,0.009034,1,1.049545,1.209194,3.522753,2
12912,7-TVD-8D-RJS,2025-10-31,Rio de Janeiro,Campos,2915.1964,1605.8897,20.1178,611.0,26.80,1.987930,...,910.0,TARTARUGA VERDE,853,2016.3880,0.006435,1,1.049545,1.209194,3.522753,2
12913,7-TVD-8D-RJS,2025-11-30,Rio de Janeiro,Campos,3009.8116,1658.0103,19.2372,584.0,26.80,1.987930,...,910.0,TARTARUGA VERDE,883,1930.0290,0.003925,1,1.049545,1.209194,3.522753,2


In [94]:
X_train = train.drop([target_column] + ignore_columns, axis=1)
X_test = test.drop([target_column] + ignore_columns, axis=1)

y_train = train[target_column]
y_test = test[target_column]

In [95]:
# train = sample_df[~sample_df['Nome_Poco_ANP'].isin(pocos_teste)]
# test = sample_df[sample_df['Nome_Poco_ANP'].isin(pocos_teste)]

# X_train = train.drop([target_column] + ignore_columns, axis=1)
# y_train = train[target_column]

# X_test = test.drop([target_column] + ignore_columns, axis=1)
# y_test = test[target_column]

In [96]:
# X = sample_df.drop(ignore_columns + [target_column], axis=1)
# y = sample_df[target_column]
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.18, random_state=42)

In [97]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Evaluate Methods

In [98]:
results_df = pd.DataFrame()

def evaluate_and_record(model_name, model, X_train, y_train, X_test, y_test, best_params=None):
    global results_df
    
    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()
    training_time = end_time - start_time
    
    y_pred_treino = model.predict(X_train)
    y_pred_teste = model.predict(X_test)
    
    rmse_treino = np.sqrt(mean_squared_error(y_train, y_pred_treino))
    r2_treino = r2_score(y_train, y_pred_treino)
    mae_treino = mean_absolute_error(y_train, y_pred_treino)
    
    rmse_teste = np.sqrt(mean_squared_error(y_test, y_pred_teste))
    r2_teste = r2_score(y_test, y_pred_teste)
    mae_teste = mean_absolute_error(y_test, y_pred_teste)
    
    new_row = {
        'Modelo': model_name,
        'R2 Treino': r2_treino,
        'R2 Teste': r2_teste,
        'RMSE Treino': rmse_treino,
        'RMSE Teste': rmse_teste,
        'MAE Treino': mae_treino,
        'MAE Teste': mae_teste,
        'Tempo (segundos)': training_time,
        'Hyperparametros': str(best_params) if best_params else 'N/A'
    }
    
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
    return model

In [99]:
def record_gridsearch_details(grid_search_object, model_prefix):
    global results_df
    
    results = pd.DataFrame(grid_search_object.cv_results_)
    
    for index, row in results.iterrows():
        mean_r2_cv = row['mean_test_score']
        
        mean_fit_time = row['mean_fit_time']
        
        model_name_detail = f"{model_prefix} - Combinacao {index + 1}"
        
        new_row = {
            'Modelo': model_name_detail,
            'R2 Teste': mean_r2_cv,
            'RMSE Teste': np.nan,
            'Tempo (s)': mean_fit_time,
            'Detalhes': f"Combin. Testada (Rank: {row['rank_test_score']})",
            'Hyperparametros': str(row['params'])
        }
        
        results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

In [100]:
cv_strategy = KFold(n_splits=3, shuffle=True, random_state=42)

# LinearRegression

In [101]:
model = LinearRegression()
evaluate_and_record("Regressao Linear", model, X_train_scaled, y_train, X_test_scaled, y_test)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](14,)","[-108.16,1356.79, 393.8 ,..., 11.82, -97.37,-292.02]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,2494
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,14
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(14)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](14,)","[192.43,152.89,137. ,..., 36.63, 23.8 , 0.44]"


# Ridge

In [102]:
scoring_metric = 'neg_mean_squared_error'

ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=42))
])

alphas = np.logspace(-4, 2, 30)

param_grid_ridge = {
    'ridge__alpha': alphas
}

# 4. GridSearchCV
grid_search_ridge = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid_ridge,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_ridge.fit(X_train, y_train)

best_alpha_ridge = grid_search_ridge.best_params_['ridge__alpha']
best_ridge_model = grid_search_ridge.best_estimator_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


In [103]:
evaluate_and_record('Ridge Otimo', best_ridge_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_ridge)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value
"mean_ mean_: ndarray of shape (n_features,) or NoneThe mean value for each feature in the training set.Equal to ``None`` when ``with_mean=False`` and ``with_std=False``.","ndarray[float64](14,)","[ 0., 0., 0.,...,-0., 0., 0.]"


# Lasso

In [104]:
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(random_state=42, max_iter=2000))
])

param_grid_lasso = {
    'lasso__alpha': alphas
}

grid_search_lasso = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=param_grid_lasso,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_lasso.fit(X_train, y_train)

best_alpha_lasso = grid_search_lasso.best_params_['lasso__alpha']
best_lasso_model = grid_search_lasso.best_estimator_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


d:\Codigos\MMQ_MSC_Icaro\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.292919e+09, tolerance: 4.275e+06
  model = cd_fast.enet_coordinate_descent(


In [105]:
evaluate_and_record('Lasso Otimo', best_lasso_model, X_train_scaled, y_train, X_test_scaled, y_test, best_alpha_lasso)

d:\Codigos\MMQ_MSC_Icaro\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.292919e+09, tolerance: 4.275e+06
  model = cd_fast.enet_coordinate_descent(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('lasso', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value
"mean_ mean_: ndarray of shape (n_features,) or NoneThe mean value for each feature in the training set.Equal to ``None`` when ``with_mean=False`` and ``with_std=False``.","ndarray[float64](14,)","[ 0., 0., 0.,...,-0., 0., 0.]"


In [106]:
results_df

,Modelo,R2 Treino,R2 Teste,RMSE Treino,RMSE Teste,MAE Treino,MAE Teste,Tempo (segundos),Hyperparametros
0,Regressao Linear,0.612694,0.404969,1278.902130,1477.089809,837.373406,862.588734,0.024822,N/A
1,Ridge Otimo,0.612694,0.404967,1278.902149,1477.093006,837.367898,862.575228,0.004627,0.0006723357536499335
2,Lasso Otimo,0.611849,0.403153,1280.295594,1479.342293,837.657425,860.712089,3.109926,0.0001


# Random Forest

In [107]:
from sklearn.ensemble import RandomForestRegressor

In [108]:
rf_model = RandomForestRegressor(random_state=42, n_jobs=1)

param_grid_rf = {
    'n_estimators': [400, 500, 600],
    'min_samples_split': [2, 5, 8],
    'min_samples_leaf': [1, 2, 4],
    'max_depth': [None, 10, 20],
    'bootstrap': [True, False]
}

# 4. GridSearchCV
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid_rf,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

best_rf_model = grid_search_rf.best_estimator_
best_params_rf = grid_search_rf.best_params_

best_n_estimators_rf = best_params_rf['n_estimators']
best_min_samples_split_rf = best_params_rf['min_samples_split']
best_min_samples_leaf_rf = best_params_rf['min_samples_leaf']
best_bootstrap_rf = best_params_rf['bootstrap']

Fitting 3 folds for each of 162 candidates, totalling 486 fits


In [109]:
evaluate_and_record('Random Forest Otimo', best_rf_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_rf.best_params_)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",600
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"ma

# Gradient Boosting

In [110]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [111]:
from xgboost import XGBRegressor

In [112]:
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_jobs=1
)

param_grid_xgb = {
    'n_estimators': [1000, 2000, 3000],
    'learning_rate': [0.01, 0.025, 0.05],
    'max_depth': [3, 6, 9],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.8, 1.0]
}

grid_search_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid_xgb,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

# 5. Fit com X_train original (sem necessidade de StandardScaler)
grid_search_xgb.fit(X_train, y_train)

# Resultados
best_xgb_model = grid_search_xgb.best_estimator_
best_params_xgb = grid_search_xgb.best_params_

Fitting 3 folds for each of 162 candidates, totalling 486 fits


In [113]:
evaluate_and_record('Gradient Boosting Otimo', best_xgb_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_xgb.best_params_)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'rmse'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


# KNN Regression

In [114]:
from sklearn.neighbors import KNeighborsRegressor

In [115]:
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

grid_search_knn = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid_knn,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_knn.fit(X_train, y_train)

best_knn_model = grid_search_knn.best_estimator_
best_params_knn = grid_search_knn.best_params_

best_n_neighbors_knn = best_params_knn['knn__n_neighbors']
best_weights_knn = best_params_knn['knn__weights']
best_p_knn = best_params_knn['knn__p']

Fitting 3 folds for each of 24 candidates, totalling 72 fits


In [116]:
evaluate_and_record('KNN Otimo', best_knn_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_knn.best_params_)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('knn', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value
"mean_ mean_: ndarray of shape (n_features,) or NoneThe mean value for each feature in the training set.Equal to ``None`` when ``with_mean=False`` and ``with_std=False``.","ndarray[float64](14,)","[ 0., 0., 0.,...,-0., 0., 0.]"


# SVR (Suport Vector Regression)

In [117]:
from sklearn.svm import SVR

In [118]:
svr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

param_grid_svr = [
    {
        'svr__kernel': ['rbf'],
        'svr__C': [0.1, 1, 10, 100, 1000],
        'svr__gamma': ['scale', 'auto'],
        'svr__epsilon': [0.01, 0.1, 0.5, 1.0]
    },
    {
        'svr__kernel': ['linear'],
        'svr__C': [0.1, 1, 10, 100],
        'svr__epsilon': [0.01, 0.1, 0.5, 1.0]
    },
    {
        'svr__kernel': ['poly', 'sigmoid'],
        'svr__C': [0.1, 1, 10, 100],
        'svr__gamma': ['scale', 'auto'],
        'svr__epsilon': [0.01, 0.1, 0.5],
        'svr__coef0': [0.0, 0.1]
    }
]

grid_search_svr = GridSearchCV(
    estimator=svr_pipeline,
    param_grid=param_grid_svr,
    scoring=scoring_metric,
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_svr.fit(X_train, y_train)

best_svr_model = grid_search_svr.best_estimator_
best_params_svr = grid_search_svr.best_params_

best_C_svr = best_params_svr.get('svr__C')
best_epsilon_svr = best_params_svr.get('svr__epsilon')
best_kernel_svr = best_params_svr.get('svr__kernel')
best_gamma_svr = best_params_svr.get('svr__gamma')
best_coef0_svr = best_params_svr.get('svr__coef0')

Fitting 3 folds for each of 152 candidates, totalling 456 fits


In [119]:
evaluate_and_record('SVR Otimo', best_svr_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_svr.best_params_)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value
"mean_ mean_: ndarray of shape (n_features,) or NoneThe mean value for each feature in the training set.Equal to ``None`` when ``with_mean=False`` and ``with_std=False``.","ndarray[float64](14,)","[ 0., 0., 0.,...,-0., 0., 0.]"


# LightGBM

In [120]:
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [121]:
from lightgbm import LGBMRegressor

In [122]:
lgbm_model = LGBMRegressor(
    objective='regression',
    random_state=42,
    n_jobs=1,
    verbose=-1
)

param_grid_lgbm = {
    'n_estimators': [500, 1000, 2000],
    'learning_rate': [0.01, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'max_depth': [-1, 5, 8],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

grid_search_lgbm = GridSearchCV(
    estimator=lgbm_model,
    param_grid=param_grid_lgbm,
    scoring='neg_mean_squared_error',
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1
)

grid_search_lgbm.fit(X_train, y_train)

# Resultados
best_lgbm_model = grid_search_lgbm.best_estimator_
best_params_lgbm = grid_search_lgbm.best_params_

Fitting 3 folds for each of 729 candidates, totalling 2187 fits


In [123]:
evaluate_and_record('LightGBM Otimo', best_lgbm_model, X_train_scaled, y_train, X_test_scaled, y_test, grid_search_lgbm.best_params_)

,learning_rate,0.05
,n_estimators,2000
,objective,'regression'
,subsample,0.7
,colsample_bytree,0.7
,random_state,42
,n_jobs,1
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1


# GridSearch Details

In [124]:
record_gridsearch_details(grid_search_ridge, "Ridge GridSearch")
record_gridsearch_details(grid_search_lasso, "Lasso GridSearch")
record_gridsearch_details(grid_search_rf, "Random Forest GridSearch")
record_gridsearch_details(grid_search_xgb, "XGBoost GridSearch")
record_gridsearch_details(grid_search_knn, "KNN GridSearch")
record_gridsearch_details(grid_search_svr, "SVR GridSearch")
record_gridsearch_details(grid_search_lgbm, "LightGBM GridSearch") 

In [125]:
results_df

,Modelo,R2 Treino,R2 Teste,RMSE Treino,RMSE Teste,MAE Treino,MAE Teste,Tempo (segundos),Hyperparametros,Tempo (s),Detalhes
0,Regressao Linear,0.612694,0.404969,1278.902130,1477.089809,837.373406,862.588734,0.024822,N/A,NaN,NaN
1,Ridge Otimo,0.612694,0.404967,1278.902149,1477.093006,837.367898,862.575228,0.004627,0.0006723357536499335,NaN,NaN
2,Lasso Otimo,0.611849,0.403153,1280.295594,1479.342293,837.657425,860.712089,3.109926,0.0001,NaN,NaN
3,Random Forest Otimo,0.992432,0.814454,178.768011,824.828274,107.833115,543.448934,28.683769,"{'bootstrap': True, 'max_depth': None, 'min_sa...",NaN,NaN
4,Gradient Boosting Otimo,0.997970,0.834680,92.598428,778.574707,69.155092,519.523924,5.483740,"{'colsample_bytree': 0.8, 'learning_rate': 0.0...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1292,LightGBM GridSearch - Combinacao 725,NaN,-178313.500851,NaN,NaN,NaN,NaN,NaN,"{'colsample_bytree': 1.0, 'learning_rate': 0.0...",2.879853,Combin. Testada (Rank: 61)
1293,LightGBM GridSearch - Combinacao 726,NaN,-178313.500851,NaN,NaN,NaN,NaN,NaN,"{'colsample_bytree': 1.0, 'learning_rate': 0.0...",2.928257,Combin. Testada (Rank: 61)
1294,LightGBM GridSearch - Combinacao 727,NaN,-192781.562514,NaN,NaN,NaN,NaN,NaN,"{'colsample_bytree': 1.0, 'learning_rate': 0.0...",3.645875,Combin. Testada (Rank: 169)
1295,LightGBM GridSearch - Combinacao 728,NaN,-192781.562514,NaN,NaN,NaN,NaN,NaN,"{'colsample_bytree': 1.0, 'learning_rate': 0.0...",3.452264,Combin. Testada (Rank: 169)


In [126]:
results_df.to_csv('model_comparison_results.csv', index=False)